<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Reference Agent System Walkthrough</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>



**Purpose:**  
This notebook presents a complete, realistic example of a working AI Agent System.  

Instead of focusing on single concepts, we will walk through an entire agent system from architecture to execution, so you can see how all the pieces (tools, dispatcher, prompt, reasoning, and error handling) come together in practice.

**Real-World Scenario:**  

We will build a **Medical Information Assistant** that can answer general health-related queries responsibly by using multiple tools, showing clear reasoning, and always including medical disclaimers.

**Important Note:**  
This is for educational purposes only. Real medical advice should always come from qualified healthcare professionals.

## 1. System Architecture Overview

Our Reference Medical Agent System follows a clean, modular architecture:

**Core Components:**

- **Tools Layer**: Specialized tools that fetch reliable information (disease/symptom info, drug details, BMI calculation, etc.).
- **Tool Registry**: Central store that holds all available tools for easy management and lookup.
- **Tool Dispatcher**: Safe execution layer that routes requests to the correct tool and handles errors gracefully.
- **LLM + ReAct Prompt**: The reasoning engine that decides the sequence of tool calls.
- **AgentExecutor**: Orchestrates the full multi-step reasoning loop.

**Data Flow:**
1. User asks a health-related question
2. Agent reasons step-by-step and selects appropriate tools
3. Tool Dispatcher executes the tools safely
4. Results are fed back to the agent
5. Agent synthesizes information and provides a final answer with disclaimers

This architecture is widely used in real-world agent systems because it is scalable, maintainable, and auditable.

## 1. Setup

In [1]:
!pip install -q --force-reinstall "langchain>=0.1.0,<1.0" "langchain-core==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" langsmith duckduckgo-search ddgs faiss-cpu beautifulsoup4 requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.6/744.6 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18

In [2]:
import os
from getpass import getpass
import requests

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete")

Enter your OpenAI API key: ··········
Setup complete


In [3]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup

## 3. Tool Definitions

We define four meaningful tools for our Medical Information Assistant:

- Real symptom/disease information from Wikipedia
- Basic drug information
- BMI calculation
- Mandatory medical disclaimer tool

In [8]:
@tool
def get_medical_info(condition: str) -> str:
    """Fetch reliable medical information about a condition or symptom from Wikipedia."""
    try:
        url = f"https://en.wikipedia.org/wiki/{condition.replace(' ', '_')}"
        headers = {"User-Agent": "Mozilla/5.0 (compatible; AI-Agent-Demo/1.0)"}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        summary = ""
        for p in soup.find_all("p")[:3]:
            text = p.get_text(strip=True)
            if len(text) > 100:
                summary += text + "\n\n"

        return f"Information about '{condition}':\n\n" + summary[:800] + "...\n\n(Source: Wikipedia)"
    except Exception as e:
        return f"Could not find reliable information for '{condition}'. Error: {str(e)}"

@tool
def get_drug_info(drug_name: str) -> str:
    """Get basic information about a common medication."""
    drug_db = {
        "paracetamol": "Used for pain relief and fever reduction. Common brand: Panadol.",
        "ibuprofen": "NSAID used for pain, inflammation, and fever.",
        "amoxicillin": "Antibiotic used to treat bacterial infections."
    }
    info = drug_db.get(drug_name.lower(), f"Basic information about {drug_name} is not available in the demo database.")
    return f"Drug Information: {info}"

@tool
def calculate_bmi(weight_kg: float, height_cm: float) -> str:
    """Calculate Body Mass Index (BMI) and provide basic interpretation."""
    bmi = weight_kg / ((height_cm / 100) ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    return f"BMI: {bmi:.1f} ({category})"

@tool
def medical_disclaimer() -> str:
    """Always return a standard medical disclaimer."""
    return """ MEDICAL DISCLAIMER: This information is for educational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment. Always consult a qualified healthcare provider."""

## 4. Tool Registry and Tool Dispatcher

The Tool Registry and Dispatcher form the backbone of our agent system, ensuring safe and organized tool execution.

In [5]:
tool_registry = {
    "get_medical_info": get_medical_info,
    "get_drug_info": get_drug_info,
    "calculate_bmi": calculate_bmi,
    "medical_disclaimer": medical_disclaimer
}

def tool_dispatcher(tool_name: str, **kwargs):
    """Safe tool dispatcher with standardized response format."""
    if tool_name not in tool_registry:
        return {"status": "error", "tool": tool_name, "error": f"Tool '{tool_name}' not found."}

    try:
        tool = tool_registry[tool_name]
        result = tool.run(kwargs) if hasattr(tool, 'run') else tool(**kwargs)
        return {"status": "success", "tool": tool_name, "result": result}
    except Exception as e:
        return {"status": "error", "tool": tool_name, "error": str(e)}

## 5. Full ReAct Agent Setup

In [6]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langsmith import Client

client = Client()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

tools = [get_medical_info, get_drug_info, calculate_bmi, medical_disclaimer]

agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=15,
    handle_parsing_errors=True
)

print("Medical Information Assistant Agent is ready")

Medical Information Assistant Agent is ready


## 6. Full Execution Trace Example

Let's run a realistic medical query and observe the complete multi-step reasoning.

In [9]:
query = "I have been having frequent headaches and mild fever for the last 3 days. What could be the possible causes and what should I do?"

response = executor.invoke({"input": query})

print("\n=== FINAL ANSWER FROM THE AGENT ===")
print(response["output"])



> Entering new AgentExecutor chain...
I need to gather reliable medical information about headaches and fever to understand the possible causes and recommended actions.  
Action: get_medical_info  
Action Input: "headache and fever"  Information about 'headache and fever':

...

(Source: Wikipedia)I need to analyze the information I received about headaches and fever to identify possible causes and recommended actions.  
Action: medical_disclaimer  
Action Input: ""   MEDICAL DISCLAIMER: This information is for educational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment. Always consult a qualified healthcare provider.I now know the final answer.  
Final Answer: Frequent headaches and mild fever can be caused by various conditions, including viral infections, sinusitis, or other illnesses. It's important to monitor your symptoms and consult a healthcare provider for a proper diagnosis and treatment. Remember, this information is for educa

## 7. Explanation of Tool Usage and Trace Analysis

In a typical execution, the agent follows this logical flow:

1. **Thought**: Understands the symptoms and decides to gather reliable medical information.
2. **Action**: Calls `get_medical_info` with the symptoms.
3. **Observation**: Receives information about possible causes.
4. **Thought**: May also call `medical_disclaimer` to ensure responsible response.
5. **Final Answer**: Provides helpful, balanced information with strong disclaimers.

This demonstrates true multi-step reasoning, responsible AI behavior, and safe tool usage.

**Key Observations:**
- The agent does not give direct medical advice
- It uses tools in a logical sequence
- It always includes disclaimers
- Real data is fetched from open sources (Wikipedia + Open-Meteo in previous tools)

## Conclusion

This Reference Agent System Walkthrough in the medical domain showed a complete, responsible, and production-like AI agent architecture.

This example demonstrates best practices for building meaningful and safe AI agents.

**Next Step:** Try modifying the query or adding new tools (e.g., vaccine information, allergy checker) to see how the system adapts.